### ספריות


In [1]:
import pandas as pd
import numpy as np
import os
import sys

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

### העלת משתנים להרצת הקוד


In [2]:
# מיקום תיקייה נוכחית
cwd = os.getcwd()

education_folder_path = os.path.dirname(cwd)

In [3]:
# תאריך
file_date=pd.Timestamp.today().strftime('%y%m%d')

### פונקציות גלובליות


In [4]:
# הוספת נתיב modules כנתיב יחסי
sys.path.append('../modules')

from global_functions import remove_spaces_in_columns, up_load_df

### העלאת טבלאות


In [5]:
# בתי ספר וגנים מעיריית ירושלים
JLM=up_load_df(r'{}\background_files\jerusalem_muni'.format(education_folder_path),'מוסדות בירושלים 2020')
JLM=remove_spaces_in_columns(JLM)

In [6]:
# קוארדינטות של בתי ספר בירושלים ממשרד החינוך
JLM_moe=up_load_df(r'{}\Intermediates'.format(cwd),'250121_JLM_moe_mosdot_coordinates_2022')
JLM_moe=remove_spaces_in_columns(JLM_moe)

In [7]:
# התאמת שמות של עמודות
JLM.rename(columns={'סמל_חינוך': 'סמל_מוסד'}, inplace=True)

# מחיקת שורות של גני ילדים
JLM=JLM[JLM['שלב_חינוך'] != 'גני ילדים']

# מחיקת עמודות מיותרות
JLM = JLM.drop(columns=['סמל_עירייה', 'אגף', 'שלב_חינוך', 'סוג_חינוך', 'סוג_חינוך', 'מכיתה', 'עד_כיתה', 'מספר_כיתות', 'סה"כ_תלמידים', 'פיקוח', 'מעמד_משפטי', "מס'_תלמידים_ז-ט", "מס'_תלמידים_י-יד", 'קוד_אזור_סטיסטי', 'תאור_אזור_סטטיסטי', 'Unnamed:_18', 'Unnamed:_19'])

In [8]:
JLMNaN = JLM[JLM['coordinate_x'].isna() | JLM['coordinate_y'].isna()]
JLMNaN = JLM[(JLM['coordinate_x'] == 0) | (JLM['coordinate_y'] == 0)]
JLM = JLM[~JLM['coordinate_x'].isna() | ~JLM['coordinate_y'].isna()]
JLM = JLM[~(JLM['coordinate_x'] == 0) | ~(JLM['coordinate_y'] == 0)]

In [9]:
# JLM_moe
filtered_JLM_moe = JLM_moe[JLM_moe['סמל_מוסד'].isin(JLMNaN['סמל_מוסד'])]

In [10]:
# מיזוג הטבלאות
JLM_moe_coordinates = pd.merge(
    JLMNaN,
    filtered_JLM_moe,
    on='סמל_מוסד',
    suffixes=('_JLMNaN', '_filtered_JLM_moe')  # מוסיף סיומות לשמות עמודות זהים
)

# בחירת עמודות מטבלה מסוימת
JLM_moe_coordinates['coordinate_x'] = JLM_moe_coordinates['coordinate_x_filtered_JLM_moe']
JLM_moe_coordinates['coordinate_y'] = JLM_moe_coordinates['coordinate_y_filtered_JLM_moe']

# מחיקת העמודות המיותרות
JLM_moe_coordinates = JLM_moe_coordinates.drop(columns=['יישוב', 'כתובת', 'שם_מוסד_filtered_JLM_moe' , 'coordinate_x_JLMNaN', 'coordinate_y_JLMNaN', 'coordinate_x_filtered_JLM_moe', 'coordinate_y_filtered_JLM_moe'])
JLM_moe_coordinates.rename(columns={'שם_מוסד_JLMNaN': 'שם_מוסד'}, inplace=True)

JLM_moe_coordinates.to_excel(r'{}\background_files\JTMT_setls_schools_coordinates_with_src\{}_JLM_moe_coordinates.xlsx'.format(education_folder_path, file_date), index=False)

### עיבוד


In [11]:
# הגדרת מקור קוארדינטות ל- jerusalem_muni
JLM['SRC'] = 'jerusalem_muni'

### ייצוא


In [12]:
JLM.to_excel(r'{}\background_files\JTMT_setls_schools_coordinates_with_src\{}_JLM_muni_coordinates.xlsx'.format(education_folder_path, file_date), index=False)